In [56]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [57]:
df=pd.read_csv('train.txt',sep=';',header=None,names=['text','emotion'])

In [58]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [59]:
df.isnull().sum()

,0
text,0
emotion,0


In [60]:
unique_emotion=df['emotion'].unique()
emotion_number={}
i=0
for emo in unique_emotion:
  emotion_number[emo]=i
  i+=1

df['emotion']=df['emotion'].map(emotion_number)

In [61]:
unique_emotion
emotion_number

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}

In [62]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [63]:
#lower case conversion
df['text']=df['text'].apply(lambda x : x.lower())

In [64]:
#remove punctuation
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [65]:
 df['text']=df['text'].apply(remove_punc)

In [66]:
#remove numbers
def remove_number(txt):
  new=""
  for i in txt:
    if not i.isdigit():
      new=new+i
  return new

df['text']=df['text'].apply(remove_number)

In [67]:
# print(remove_number('mansi123 plaiwal'))


In [68]:
#remove emojis
def remove_emoji(txt):
  new=""
  for i in txt:
    if i.isascii():
      new=new+i
  return new

df['text']=df['text'].apply(remove_emoji)

In [69]:
#remove stop words (is,am,are,for common word to reduce noise)
import nltk

In [70]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [71]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [72]:
stop_words=set(stopwords.words('english'))
# stop_words

In [73]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [74]:
#remove stopwords
def remove(txt):
  words=txt.split()
  cleaned=[]
  for i in words:
    if not i in stop_words:
      cleaned.append(i)
  return ' '.joined(cleaned)

below code is just for understanding purpose

In [75]:
# one hot encoding and bagging
from sklearn.feature_extraction.text import CountVectorizer

document=[
    "I love pizza",
    "pizza is the best",
    "I love pasta",
    "pizza is great"
]

vectorizer=CountVectorizer() #(ngram_range=(1,3))

X=vectorizer.fit_transform(document)

print("vocab:",vectorizer.get_feature_names_out())
print(X.toarray())

vocab: ['best' 'great' 'is' 'love' 'pasta' 'pizza' 'the']
[[0 0 0 1 0 1 0]
 [1 0 1 0 0 1 1]
 [0 0 0 1 1 0 0]
 [0 1 1 0 0 1 0]]


In [76]:
from sklearn.feature_extraction.text import TfidfVectorizer

document=[
    "I love pizza",
    "pizza is the best",
    "I love pasta",
    "pizza is great"
]

vectorizer=TfidfVectorizer() #(ngram=(1,3))

X=vectorizer.fit_transform(document)

print("vocab:",vectorizer.get_feature_names_out())
print(X.toarray())

vocab: ['best' 'great' 'is' 'love' 'pasta' 'pizza' 'the']
[[0.         0.         0.         0.77722116 0.         0.62922751
  0.        ]
 [0.57457953 0.         0.4530051  0.         0.         0.36674667
  0.57457953]
 [0.         0.         0.         0.6191303  0.78528828 0.
  0.        ]
 [0.         0.70203482 0.55349232 0.         0.         0.44809973
  0.        ]]


In [77]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.2, random_state=42)

In [78]:
X_train

,text
676,i refers of course though i cant help feeling ...
12113,im starting to feel that im suffering from fat...
7077,i feel like i probably would have liked this b...
13005,i didn t really feel awkward at all
12123,im feeling a little grumpy today with the lame...
...,...
13418,i love this because to me it should leave the ...
5390,i feel is very delicate
860,i was starting to feel a little stressed
15795,i feel stressed tired worn out out of shape or...


In [79]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

In [80]:
bow_vectorizer=CountVectorizer()
X_train_bow=bow_vectorizer.fit_transform(X_train)
X_test_bow=bow_vectorizer.transform(X_test)

In [81]:
# X_train_bow
from sklearn.naive_bayes import MultinomialNB
model_nb=MultinomialNB()

In [82]:
from sklearn.metrics import accuracy_score

In [83]:
model_nb.fit(X_train_bow,y_train)

MultinomialNB()

In [84]:
X_test_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 47793 stored elements and shape (3200, 13501)>

In [85]:
y_pred_bow=model_nb.predict(X_test_bow)

In [86]:
accuracy=accuracy_score(y_test,y_pred_bow) ##bow techniques
print(accuracy)

0.7390625


In [87]:
#tfidf technique
tfidf_vectorizer=TfidfVectorizer()
X_train_tf=tfidf_vectorizer.fit_transform(X_train)
X_test_tf=tfidf_vectorizer.transform(X_test)

model_nb2=MultinomialNB()


In [88]:
model_nb2.fit(X_train_tf,y_train)

MultinomialNB()

In [89]:
y_pred_tf=model_nb2.predict(X_test_tf)

In [90]:
accuracy=accuracy_score(y_test,y_pred_tf) ##bow techniques
print(accuracy)

0.6175


In [91]:
#logistic regression
from sklearn.linear_model import LogisticRegression
model_lr=LogisticRegression(max_iter=1000)

model_lr.fit(X_train_tf,y_train)

LogisticRegression(max_iter=1000)

In [92]:
y_pred_lr=model_lr.predict(X_test_tf)

In [93]:
accuracy=accuracy_score(y_test,y_pred_lr) ##bow techniques
print(accuracy)

0.84125


In [94]:
import joblib
joblib.dump(model_lr,'model.pkl')

['model.pkl']

In [95]:
joblib.dump(tfidf_vectorizer,'columns.pkl')

['columns.pkl']